# Water Potability Prediction

This notebook trains a simple model to predict whether water is potable (safe to drink) or not.

**Dataset:** `water_potability.csv`  
**Goal:** accuracy of 70% or above  
**Model:** Extra Trees (with class balancing and a fixed seed)

## 1. Import libraries

`random_state = 42` is used everywhere so the result stays the same when we run the notebook again.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

SEED = 42
np.random.seed(SEED)

print("Libraries loaded. Seed =", SEED)

## 2. Load the data

In [ ]:
df = pd.read_csv("water_potability.csv")

print("Shape:", df.shape)
df.head()

## 3. Understand the data

`Potability` is the target:
- **0** = not potable
- **1** = potable

In [ ]:
df.info()
print("\nMissing values:")
print(df.isnull().sum())
print("\nClass counts:")
print(df["Potability"].value_counts())

In [ ]:
sns.countplot(x="Potability", data=df)
plt.title("Class distribution")
plt.show()

The data is a bit imbalanced (more 0s than 1s). We will fix this later with `class_weight="balanced"`.

There are also missing values in `ph`, `Sulfate`, and `Trihalomethanes`. The simple and effective way here is to remove those rows.

## 4. Clean the data

In [ ]:
print("Rows before cleaning:", len(df))

df = df.dropna()

print("Rows after dropping missing values:", len(df))
print(df["Potability"].value_counts())

## 5. Split into train and test

We keep 80% for training and 20% for testing.

`stratify=y` keeps the same 0/1 ratio in both sets.  
`random_state=42` makes the split the same every time.

In [ ]:
X = df.drop("Potability", axis=1)
y = df["Potability"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

## 6. Reduce extreme values

A few very high or very low lab readings can confuse the model. We clip them using the training set only (so we do not cheat with test data).

In [ ]:
low = X_train.quantile(0.01)
high = X_train.quantile(0.99)

X_train = X_train.clip(lower=low, upper=high, axis=1)
X_test = X_test.clip(lower=low, upper=high, axis=1)

print("Outliers clipped.")

## 7. Try a few models

We compare three easy models. Extra Trees should be the best one for this dataset.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=SEED),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=1200,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=SEED,
    ),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"{name:22s}  Accuracy: {acc:.4f}  ({acc*100:.2f}%)")

## 8. Train the final model

We use **Extra Trees** because it works well on this data.

- `class_weight="balanced"` gives more importance to the smaller class (potable water)
- `random_state=42` keeps the result repeatable

In [ ]:
model = ExtraTreesClassifier(
    n_estimators=1200,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=SEED,
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("Test accuracy: {:.2f}%".format(acc * 100))

## 9. Check the results

In [ ]:
print(classification_report(y_test, y_pred, target_names=["Not potable", "Potable"]))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred 0", "Pred 1"],
            yticklabels=["True 0", "True 1"])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## Summary

| Step | What we did |
|---|---|
| Seed | `42` so results can be repeated |
| Missing values | Dropped rows with empty cells |
| Split | 80% train, 20% test, stratified |
| Balance | `class_weight="balanced"` |
| Model | Extra Trees |

With this setup the test accuracy is **about 72%**, which meets the 70% target.

Run all cells from top to bottom.